In [1]:
import glob
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 1. LOAD & CONCATENATE ALL 12 CSV FILES (2025 DATA)
# -------------------------------------------------------------
# Adjust the path to match where your 12 monthly CSV files live
all_files = sorted(glob.glob("data/*.csv"))
print(f"Found {len(all_files)} monthly files. Concatenating...")

# Read and combine all monthly files into one master DataFrame
df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
print(f"Initial raw rows loaded: {len(df):,}")

# -------------------------------------------------------------
# 2. FEATURE ENGINEERING (CASE STUDY REQUIREMENTS)
# -------------------------------------------------------------
# Convert timestamp strings to pandas datetime objects
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

# Calculate ride_length in minutes (float)
df["ride_length_min"] = (
    df["ended_at"] - df["started_at"]
).dt.total_seconds() / 60

# Extract day of week name and month name
df["day_of_week"] = df["started_at"].dt.day_name()
df["month"] = df["started_at"].dt.month_name()

# Order days chronologically for consistent plotting/tables
days_order = [
    "Sunday",
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
]
df["day_of_week"] = pd.Categorical(
    df["day_of_week"], categories=days_order, ordered=True
)

# -------------------------------------------------------------
# 3. DATA CLEANING & OUTLIER REMOVAL
# -------------------------------------------------------------
# Filter 1: Drop false docks / negative durations (trips <= 1 minute)
# Filter 2: Drop maintenance/stolen bike anomalies (trips >= 24 hours / 1440 min)
clean_condition = (df["ride_length_min"] > 1) & (df["ride_length_min"] < 1440)
df_clean = df[clean_condition].copy()

# Remove duplicate ride IDs if any exist
df_clean.drop_duplicates(subset=["ride_id"], inplace=True)

print(f"Cleaned valid rows remaining: {len(df_clean):,}")
print(f"Removed {len(df) - len(df_clean):,} invalid or outlier records.")

# -------------------------------------------------------------
# 4. DESCRIPTIVE ANALYSIS (CORE METRICS)
# -------------------------------------------------------------
print("\n--- OVERALL DURATION SUMMARY (MINUTES) ---")
print(
    f"Mean ride length   : {df_clean['ride_length_min'].mean():.2f} min"
)  # Course requirement: Mean
print(
    f"Max ride length    : {df_clean['ride_length_min'].max():.2f} min"
)  # Course requirement: Max
print(
    f"Mode day of week   : {df_clean['day_of_week'].mode()[0]}"
)  # Course requirement: Mode

# Summary Table 1: Member vs. Casual by Day of Week
weekly_summary = (
    df_clean.groupby(["member_casual", "day_of_week"], observed=True)
    .agg(
        total_rides=("ride_id", "count"),
        avg_duration_min=("ride_length_min", "mean"),
    )
    .reset_index()
)

print("\n--- WEEKLY SUMMARY TABLE ---")
print(weekly_summary)

# Summary Table 2: Bike Type Preference
bike_summary = (
    df_clean.groupby(["member_casual", "rideable_type"])
    .agg(total_rides=("ride_id", "count"))
    .reset_index()
)

print("\n--- BIKE PREFERENCE TABLE ---")
print(bike_summary)

# -------------------------------------------------------------
# 5. EXPORT AGGREGATED SUMMARIES (FOR VISUALIZATION)
# -------------------------------------------------------------
# Export the clean, small summary files for Tableau or Excel charts
weekly_summary.to_csv("cyclistic_weekly_summary.csv", index=False)
bike_summary.to_csv("cyclistic_bike_summary.csv", index=False)
print("\nExported summary CSVs successfully.")

Found 12 monthly files. Concatenating...
Initial raw rows loaded: 5,552,994
Cleaned valid rows remaining: 5,400,007
Removed 152,987 invalid or outlier records.

--- OVERALL DURATION SUMMARY (MINUTES) ---
Mean ride length   : 14.92 min
Max ride length    : 1439.98 min
Mode day of week   : Saturday

--- WEEKLY SUMMARY TABLE ---
   member_casual day_of_week  total_rides  avg_duration_min
0         casual      Sunday       316953         23.102187
1         casual      Monday       219232         19.699367
2         casual     Tuesday       216926         17.615906
3         casual   Wednesday       212744         16.360071
4         casual    Thursday       247893         17.448413
5         casual      Friday       306463         19.599456
6         casual    Saturday       395595         22.401703
7         member      Sunday       374535         13.445723
8         member      Monday       493322         11.757136
9         member     Tuesday       552529         11.845756
10        me

In [2]:
!pip install openpyxl

In [4]:
import pandas as pd

print("Generating clean summary tables from df_clean...")

# -------------------------------------------------------------
# 1. PREPARE TIME COLUMNS
# -------------------------------------------------------------
df_clean["month_num"] = df_clean["started_at"].dt.month
df_clean["month_name"] = df_clean["started_at"].dt.month_name()
df_clean["day_of_week"] = df_clean["started_at"].dt.day_name()

# -------------------------------------------------------------
# 2. MONTHLY SEASONALITY SUMMARY
# -------------------------------------------------------------
monthly_summary = (
    df_clean.groupby(["month_num", "month_name", "member_casual"], observed=True)
    .agg(
        total_rides=("ride_id", "count"),
        avg_duration_min=("ride_length_min", "mean")
    )
    .reset_index()
)

# -------------------------------------------------------------
# 3. WEEKLY COMMUTE SUMMARY
# -------------------------------------------------------------
weekly_summary = (
    df_clean.groupby(["day_of_week", "member_casual"], observed=True)
    .agg(
        total_rides=("ride_id", "count"),
        avg_duration_min=("ride_length_min", "mean")
    )
    .reset_index()
)

# -------------------------------------------------------------
# 4. BIKE / FLEET PREFERENCE SUMMARY
# -------------------------------------------------------------
bike_summary = (
    df_clean.groupby(["rideable_type", "member_casual"], observed=True)
    .agg(
        total_rides=("ride_id", "count"),
        avg_duration_min=("ride_length_min", "mean")
    )
    .reset_index()
)

# -------------------------------------------------------------
# 5. TOP 50 START STATIONS
# -------------------------------------------------------------
station_clean = df_clean[df_clean["start_station_name"].notna()].copy()
top_stations = (
    station_clean.groupby(["member_casual", "start_station_name"], as_index=False)
    .agg(
        total_rides=("ride_id", "count"),
        avg_duration_min=("ride_length_min", "mean")
    )
    .sort_values(by="total_rides", ascending=False)
    .head(50)
)

# -------------------------------------------------------------
# 6. EXPORT TO MULTI-TAB EXCEL WORKBOOK
# -------------------------------------------------------------
excel_filename = "Cyclistic_2025_Summary_Export.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    monthly_summary.to_excel(writer, sheet_name="Monthly_Seasonality", index=False)
    weekly_summary.to_excel(writer, sheet_name="Weekly_Patterns", index=False)
    bike_summary.to_excel(writer, sheet_name="Bike_Preferences", index=False)
    top_stations.to_excel(writer, sheet_name="Top_50_Stations", index=False)

print(f"✅ Created full Excel workbook: {excel_filename}")

# -------------------------------------------------------------
# 7. ALSO EXPORT CLEAN CSVs AS BACKUP
# -------------------------------------------------------------
monthly_summary.to_csv("cyclistic_monthly_summary.csv", index=False)
weekly_summary.to_csv("cyclistic_weekly_summary.csv", index=False)
bike_summary.to_csv("cyclistic_bike_summary.csv", index=False)

print("✅ Saved all individual summary CSVs.")

Generating clean summary tables from df_clean...
✅ Created full Excel workbook: Cyclistic_2025_Summary_Export.xlsx
✅ Saved all individual summary CSVs.


In [5]:
import numpy as np
import pandas as pd

print("=" * 65)
print(" 🔍 CYCLISTIC DATA CLEANING & INTEGRITY AUDIT REPORT")
print("=" * 65)

# 1. ROW COUNT & RETENTION AUDIT
initial_rows = len(df) if "df" in locals() else None
clean_rows = len(df_clean)
print(f"1. TOTAL ROWS IN CLEAN DATASET: {clean_rows:,}")
if initial_rows:
    dropped_rows = initial_rows - clean_rows
    retention_pct = (clean_rows / initial_rows) * 100
    print(
        f"   • Raw rows: {initial_rows:,} | Filtered out: {dropped_rows:,} ({100-retention_pct:.2f}%)"
    )

# 2. DUPLICATE RIDE_ID CHECK
duplicate_ids = df_clean["ride_id"].duplicated().sum()
if duplicate_ids == 0:
    print("\n2. PRIMARY KEY INTEGRITY: ✅ PASS")
    print("   • Zero duplicate 'ride_id' records detected.")
else:
    print(f"\n2. PRIMARY KEY INTEGRITY: ❌ FAIL")
    print(f"   • Found {duplicate_ids:,} duplicate ride IDs!")

# 3. DURATION BOUNDARY CHECKS (Must be > 1 min and < 1440 min)
min_duration = df_clean["ride_length_min"].min()
max_duration = df_clean["ride_length_min"].max()
invalid_durations = (
    (df_clean["ride_length_min"] <= 1) | (df_clean["ride_length_min"] >= 1440)
).sum()

print("\n3. RIDE LENGTH BOUNDARY AUDIT:")
print(f"   • Minimum ride duration : {min_duration:.2f} minutes")
print(f"   • Maximum ride duration : {max_duration:.2f} minutes")
if invalid_durations == 0 and min_duration > 1.0 and max_duration < 1440.0:
    print("   • Duration validation   : ✅ PASS (No negative, zero, or >24h rides)")
else:
    print(
        f"   • Duration validation   : ❌ FAIL ({invalid_durations:,} records outside boundary)"
    )

# 4. MISSING VALUES & CATEGORICAL PURITY
print("\n4. CATEGORICAL VALUES AUDIT:")
member_categories = df_clean["member_casual"].unique().tolist()
bike_categories = df_clean["rideable_type"].unique().tolist()
days_present = df_clean["day_of_week"].unique().tolist()

print(f"   • User Types Found      : {member_categories}")
print(f"   • Bike Types Found      : {bike_categories}")
print(f"   • Days of Week Found    : {len(days_present)}/7 days present")

# Check for unexpected string anomalies in member_casual
if set(member_categories).issubset({"member", "casual"}):
    print("   • User Type Column      : ✅ PASS (Strictly 'member' and 'casual')")
else:
    print(
        f"   • User Type Column      : ❌ FAIL (Unexpected labels: {set(member_categories) - {'member', 'casual'}})"
    )

# 5. NULL VALUE SUMMARY ACROSS COLUMNS
print("\n5. NULL / MISSING VALUES SUMMARY:")
null_summary = df_clean.isnull().sum()
for col, val in null_summary.items():
    status = "OK" if val == 0 else f"Note: {val:,} nulls"
    print(f"   • {col:<20}: {status}")

print("\n" + "=" * 65)
print("AUDIT COMPLETE: If all checks passed, your data is certified clean.")
print("=" * 65)

 🔍 CYCLISTIC DATA CLEANING & INTEGRITY AUDIT REPORT
1. TOTAL ROWS IN CLEAN DATASET: 5,400,007
   • Raw rows: 5,552,994 | Filtered out: 152,987 (2.76%)

2. PRIMARY KEY INTEGRITY: ✅ PASS
   • Zero duplicate 'ride_id' records detected.

3. RIDE LENGTH BOUNDARY AUDIT:
   • Minimum ride duration : 1.00 minutes
   • Maximum ride duration : 1439.98 minutes
   • Duration validation   : ✅ PASS (No negative, zero, or >24h rides)

4. CATEGORICAL VALUES AUDIT:
   • User Types Found      : ['member', 'casual']
   • Bike Types Found      : ['classic_bike', 'electric_bike']
   • Days of Week Found    : 7/7 days present
   • User Type Column      : ✅ PASS (Strictly 'member' and 'casual')

5. NULL / MISSING VALUES SUMMARY:
   • ride_id             : OK
   • rideable_type       : OK
   • started_at          : OK
   • ended_at            : OK
   • start_station_name  : Note: 1,095,320 nulls
   • start_station_id    : Note: 1,095,320 nulls
   • end_station_name    : Note: 1,126,339 nulls
   • end_station_

In [6]:
import pandas as pd

# Ensure timestamp features are extracted
df_clean["hour"] = df_clean["started_at"].dt.hour
df_clean["month_num"] = df_clean["started_at"].dt.month
df_clean["month_name"] = df_clean["started_at"].dt.month_name()
df_clean["year"] = df_clean["started_at"].dt.year

# -------------------------------------------------------------
# 1. EXPORT AGGREGATED METRICS (KPIs, Trends, Heatmaps)
# -------------------------------------------------------------
# Grouping by all dashboard dimensions keeps granularity while collapsing rows
agg_tableau = (
    df_clean.groupby(
        [
            "member_casual",
            "rideable_type",
            "year",
            "month_num",
            "month_name",
            "day_of_week",
            "hour",
        ],
        observed=True,
    )
    .agg(
        total_rides=("ride_id", "count"),
        total_duration_min=("ride_length_min", "sum"),
        avg_duration_min=("ride_length_min", "mean"),
        median_duration_min=("ride_length_min", "median"),
    )
    .reset_index()
)

# Export as CSV (or .hyper if tableau-hyper-api is installed)
agg_tableau.to_csv("cyclistic_tableau_aggregated.csv", index=False)
print(f"Aggregated table exported: {len(agg_tableau):,} rows (Super fast!)")

# -------------------------------------------------------------
# 2. EXPORT TOP STATIONS & ROUTES (For Tableau Map Visuals)
# -------------------------------------------------------------
# Filter out blank stations common in electric bike dockless rides
station_df = df_clean[
    df_clean["start_station_name"].notna()
    & df_clean["end_station_name"].notna()
].copy()

# A. Start Station Volume & Locations
start_stations = (
    station_df.groupby(
        ["member_casual", "start_station_name"], as_index=False
    )
    .agg(
        station_lat=("start_lat", "mean"),
        station_lng=("start_lng", "mean"),
        total_rides=("ride_id", "count"),
        avg_duration_min=("ride_length_min", "mean"),
    )
    .sort_values(by="total_rides", ascending=False)
)

start_stations.to_csv("cyclistic_tableau_stations.csv", index=False)
print(f"Station table exported: {len(start_stations):,} rows.")

Aggregated table exported: 8,066 rows (Super fast!)
Station table exported: 3,359 rows.
